# NB01 — The Three-Pass Method and paper-notes/ Protocol

**Module:** 19 — Research Seminar and Paper Reading  
**Tier:** Ongoing/Cross-cutting  
**Estimated time:** 2 hours (notebook) + weekly practice thereafter  

---

## Motivation

Reading a scientific paper is not like reading a novel. You do not start at page 1 and read linearly to the end. Done that way, a methods-heavy computational biology paper consumes 3 hours and leaves you with a vague sense of what happened but no ability to explain it 48 hours later.

The Three-Pass Method (Keshav 2007) solves this. It is a structured, time-boxed approach that separates *understanding what a paper claims* from *understanding how it argues for those claims* from *being able to reconstruct the argument yourself*. That last pass — reconstruction with the paper closed — is the actual learning event.

More concretely: the most common bioinformatics interview pattern is being handed an abstract you have never seen and asked to explain it. The Three-Pass Method, practiced weekly for 12 months, is direct rehearsal for this.

## 1. The Three-Pass Method — Detail

### Pass 1 — Skim (5–10 minutes)

Read only:
1. Title — what is the object of study, and what is claimed about it?
2. Abstract — what problem, what approach, what finding?
3. Section headings — what is the logical structure of the paper?
4. Figures and captions — what does the paper show visually? (Captions often contain key results.)
5. Conclusion section — what do the authors say their contribution is?

**Output of Pass 1:** Write one sentence — "This paper argues that [X] by [method Y], showing [result Z]."

**Time limit:** Stop at 10 minutes even if you haven't finished. The discipline is part of the practice.

### Pass 2 — Read (45–90 minutes)

Read the full paper in order. As you read:
- Mark every claim with a `C`
- Mark every piece of evidence supporting a claim with an `E`
- Mark anything fuzzy with a `?`
- Do not stop to look everything up — mark it `?` and continue

**Output of Pass 2:** A list of the paper's main claims, their evidence, and your `?` list.

**Do not** go down citation rabbit holes. That is a different activity (building a reading list) done after Pass 2.

### Pass 3 — Reconstruct (1–2 hours)

Close the paper. Without looking:
1. State the problem the paper solves
2. Describe the method at the level of: "they took X, applied Y, and measured Z because..."
3. State the key result and what it implies
4. List two limitations the authors acknowledge and one they don't
5. Name one follow-up question this paper raises

**Log binary outcome:** Did you succeed unaided? Y / N

A `N` is not a failure — it is information. It tells you what to re-read before logging a second attempt.

## 2. The paper-notes/ Protocol

Every paper read in this programme goes into `paper-notes/` at the repository root. The log is the evidence of independent scientific engagement — for PhD applications and as an interview artifact.

### File naming convention

```
paper-notes/[firstauthor]_[year]_[short_keyword].md
```

Examples:
- `needleman_1970_sequence_alignment.md`
- `love_2014_deseq2.md`
- `jumper_2021_alphafold2.md`

### Template

```markdown
# [Author(s)] ([Year]). "[Title]"

**DOI:** ...
**Read date:** YYYY-MM-DD
**Module connection:** Module XX — [module name]

## Pass 1 — Skim (date, time spent)
One-sentence summary:

## Pass 2 — Read (date, time spent)
Key claims:
- C1: ...
- C2: ...

Evidence noted:
- C1 supported by: ...

Fuzzy points (marked ?):
- ...

## Pass 3 — Reconstruct (date, unaided)
**Success:** Y / N

Problem:
Method:
Key result:
Limitations acknowledged:
Limitations not acknowledged:
Follow-up question:

## Questions remaining
- ...
```

## 3. Python Implementation: PaperLog

The `PaperLog` class manages paper-notes/ entries and tracks Pass-3 success rate over time.

In [ ]:
import json
import os
from dataclasses import dataclass, field
from datetime import date, datetime
from typing import List, Optional


@dataclass
class PaperEntry:
    """A single paper-notes/ entry."""
    title: str
    authors: str
    year: int
    doi: str
    module: str
    pass1_date: Optional[str] = None
    pass1_summary: Optional[str] = None
    pass2_date: Optional[str] = None
    pass2_claims: List[str] = field(default_factory=list)
    pass2_fuzzy: List[str] = field(default_factory=list)
    pass3_date: Optional[str] = None
    pass3_success: Optional[bool] = None
    pass3_notes: Optional[str] = None

    def completion_level(self) -> int:
        """Return 0 (not started), 1 (Pass 1 done), 2, or 3."""
        if self.pass3_date:
            return 3
        if self.pass2_date:
            return 2
        if self.pass1_date:
            return 1
        return 0


class PaperLog:
    """Manage a collection of PaperEntry objects and compute reading statistics."""

    def __init__(self):
        self.entries: List[PaperEntry] = []

    def add(self, entry: PaperEntry) -> None:
        self.entries.append(entry)

    def pass3_success_rate(self) -> float:
        """Fraction of Pass-3 attempts that succeeded."""
        attempted = [e for e in self.entries if e.pass3_success is not None]
        if not attempted:
            return 0.0
        return sum(1 for e in attempted if e.pass3_success) / len(attempted)

    def summary(self) -> dict:
        return {
            'total_papers': len(self.entries),
            'pass1_complete': sum(1 for e in self.entries if e.completion_level() >= 1),
            'pass2_complete': sum(1 for e in self.entries if e.completion_level() >= 2),
            'pass3_attempted': sum(1 for e in self.entries if e.pass3_success is not None),
            'pass3_success_rate': round(self.pass3_success_rate(), 3),
        }

    def by_module(self) -> dict:
        """Group entries by module."""
        result = {}
        for e in self.entries:
            result.setdefault(e.module, []).append(e)
        return result

    def cumulative_pass3_rates(self) -> List[float]:
        """Return cumulative Pass-3 success rate after each attempted paper."""
        rates = []
        successes = 0
        attempts = 0
        for e in self.entries:
            if e.pass3_success is not None:
                attempts += 1
                if e.pass3_success:
                    successes += 1
                rates.append(successes / attempts)
        return rates


# --- Build a simulated 20-paper log to demonstrate the class ---
import random
random.seed(42)

log = PaperLog()

simulated_papers = [
    ('Needleman & Wunsch', 1970, 'A general method for sequence alignment', '10.1016/0022-2836(70)90057-4', 'Module 08'),
    ('Smith & Waterman', 1981, 'Identification of common molecular subsequences', '10.1016/0022-2836(81)90087-5', 'Module 08'),
    ('Altschul et al.', 1990, 'Basic local alignment search tool (BLAST)', '10.1016/S0022-2836(05)80360-2', 'Module 08'),
    ('Dobin et al.', 2013, 'STAR: ultrafast universal RNA-seq aligner', '10.1093/bioinformatics/bts635', 'Module 09'),
    ('Love et al.', 2014, 'DESeq2 moderated fold change estimation', '10.1186/s13059-014-0550-8', 'Module 10'),
    ('Li', 2013, 'BWA-MEM aligning sequence reads', 'arXiv:1303.3997', 'Module 09'),
    ('Butler et al.', 2018, 'Seurat single-cell integration', '10.1038/nbt.4096', 'Module 10'),
    ('Wolf et al.', 2018, 'SCANPY large-scale single-cell analysis', '10.1186/s13059-017-1382-0', 'Module 10'),
    ('Korsunsky et al.', 2019, 'Harmony single-cell integration', '10.1038/s41592-019-0619-0', 'Module 10'),
    ('McInnes et al.', 2018, 'UMAP uniform manifold approximation', 'arXiv:1802.03426', 'Module 13'),
    ('Jumper et al.', 2021, 'AlphaFold2 protein structure prediction', '10.1038/s41586-021-03819-2', 'Module 11'),
    ('Ioannidis', 2005, 'Why most published research findings are false', '10.1371/journal.pmed.0020124', 'Module 19'),
    ('Keshav', 2007, 'How to Read a Paper', '10.1145/1273445.1273458', 'Module 19'),
    ('Mangul et al.', 2019, 'Benchmarking omics computational tools', '10.1038/s41467-019-09406-4', 'Module 19'),
    ('Cohen', 1992, 'A power primer', '10.1037/0033-2909.112.1.155', 'Module 03'),
    ('Nuzzo', 2014, 'Statistical errors in scientific method', '10.1038/506150a', 'Module 03'),
    ('Pautasso', 2013, 'Ten simple rules for writing a literature review', '10.1371/journal.pcbi.1003149', 'Module 19'),
    ('Gentleman & Lang', 2007, 'Statistical analyses and reproducible research', '10.1198/106186007X177582', 'Module 18'),
    ('Boulesteix et al.', 2013, 'Plea for neutral comparison studies', '10.1371/journal.pone.0061562', 'Module 19'),
    ('Mensh & Kording', 2017, 'Ten simple rules for structuring papers', '10.1371/journal.pcbi.1005619', 'Module 18'),
]

# Simulate increasing Pass-3 success rate over time (learning curve)
for i, (authors, year, title, doi, module) in enumerate(simulated_papers):
    success_prob = 0.2 + 0.04 * i  # starts low, improves
    success_prob = min(success_prob, 0.85)
    passed = random.random() < success_prob
    entry = PaperEntry(
        title=title, authors=authors, year=year, doi=doi, module=module,
        pass1_date=f'2026-07-{i+1:02d}',
        pass1_summary=f'[Summary of pass 1 for {authors} ({year})]',
        pass2_date=f'2026-07-{i+1:02d}',
        pass2_claims=['Main claim 1', 'Main claim 2'],
        pass2_fuzzy=['Fuzzy term 1'],
        pass3_date=f'2026-07-{i+2:02d}',
        pass3_success=passed,
        pass3_notes='Re-explained method unaided.' if passed else 'Could not reconstruct main method — re-read methods section.',
    )
    log.add(entry)

print('Paper log summary:', json.dumps(log.summary(), indent=2))

## 4. Visualization

Three panels showing: (1) cumulative Pass-3 success rate over reading sequence, (2) time investment distribution across passes, (3) reading depth (pass level) vs. completion.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Paper Reading Progress — Three-Pass Method Tracker', fontsize=14, fontweight='bold')

# Panel 1: Cumulative Pass-3 success rate
ax1 = axes[0]
rates = log.cumulative_pass3_rates()
ax1.plot(range(1, len(rates) + 1), rates, marker='o', color='steelblue', linewidth=2, markersize=5)
ax1.axhline(y=0.65, color='green', linestyle='--', alpha=0.7, label='Month 12 target (65%)')
ax1.axhline(y=0.40, color='orange', linestyle='--', alpha=0.7, label='Month 6 target (40%)')
ax1.set_xlabel('Papers attempted (Pass 3)')
ax1.set_ylabel('Cumulative success rate')
ax1.set_title('Pass-3 Success Rate Over Time')
ax1.set_ylim(0, 1)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Panel 2: Simulated time investment distribution
ax2 = axes[1]
pass_labels = ['Pass 1\n(5–10 min)', 'Pass 2\n(45–90 min)', 'Pass 3\n(60–120 min)']
mean_times = [7.5, 67.5, 90]
std_times = [2, 15, 25]
x_pos = np.arange(len(pass_labels))
bars = ax2.bar(x_pos, mean_times, yerr=std_times, capsize=6,
               color=['#4CAF50', '#2196F3', '#FF9800'], alpha=0.85, width=0.5)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(pass_labels)
ax2.set_ylabel('Time (minutes)')
ax2.set_title('Time Investment Per Pass')
ax2.grid(True, axis='y', alpha=0.3)
total_per_paper = sum(mean_times)
ax2.text(0.5, max(mean_times) * 1.15,
         f'Total per paper: ~{total_per_paper:.0f} min (~{total_per_paper/60:.1f} hrs)',
         ha='center', va='bottom', transform=ax2.transAxes, fontsize=9)

# Panel 3: Reading depth distribution (how many at each pass level)
ax3 = axes[2]
level_counts = [0, 0, 0, 0]
for e in log.entries:
    level_counts[e.completion_level()] += 1
level_labels = ['Not started\n(0)', 'Pass 1\nonly', 'Pass 2\ncomplete', 'Pass 3\ncomplete']
colors = ['#9E9E9E', '#4CAF50', '#2196F3', '#FF9800']
bars3 = ax3.bar(level_labels, level_counts, color=colors, alpha=0.85, width=0.6)
ax3.set_ylabel('Number of papers')
ax3.set_title('Reading Depth Distribution')
ax3.grid(True, axis='y', alpha=0.3)
for bar, count in zip(bars3, level_counts):
    if count > 0:
        ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('paper_reading_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Final Pass-3 success rate:', f"{log.pass3_success_rate():.1%}")

## 5. Exercises

**Exercise 1:** Read Keshav (2007) "How to Read a Paper" (2 pages, free at the DOI). Apply Pass 1, Pass 2, and Pass 3 to it. Log in `paper-notes/keshav_2007_how_to_read.md`.

**Exercise 2:** Choose one paper from Module 08's `papers.md` (Needleman-Wunsch or Smith-Waterman). Complete all three passes. Log your Pass-3 binary outcome.

**Exercise 3:** After logging your first two papers, run `log.summary()` on your real PaperLog. What is your Pass-3 success rate? Is this expected for Week 1?

**Exercise 4 (reflection):** In one plain sentence, explain why Pass 3 (closed-paper reconstruction) is a better learning metric than "I read the paper." Do not use the word "recall" or "retention."

## Quiz (Active Recall)

1. What are the 5 things you read in Pass 1?
2. What is the output of Pass 1 — what do you write down?
3. What does the `?` marker mean in Pass 2, and what should you do when you encounter one?
4. What is the Pass-3 binary outcome, and why is `N` as valuable as `Y`?
5. Why does the `paper-notes/` log belong in the public repository rather than a private notes app?

## Reflection

Write one paragraph answering: What part of the Three-Pass Method is most unfamiliar to how I currently read papers? What specifically will I do differently starting this week?

## References

- Keshav, S. (2007). How to Read a Paper. ACM SIGCOMM CCR 37(3):83–84. DOI:10.1145/1273445.1273458

## Future work / open questions

- Does the optimal time per pass change for different paper types (methods papers vs. review papers vs. data papers)?
- How should the Three-Pass Method adapt for preprints (no peer review, results may change)?